In [0]:
%run "../../commons/commons_imports"

In [0]:
df_estado_silver = read(
    base_path=SILVER_PATH,
    table_name=TS_ESTADO,
    format="delta"
)

In [0]:
df_estado_indicador = (

    df_estado_silver

    .select(
        "ANO_REFERENCIA",
        "CO_UF",
        "ID_TIPO_REDE",
        col("PC_ALUNO_ALFABETIZADO")
            .alias("PC_ALUNO_ALFABETIZADO_ESTADO"),
        col("VL_MEDIA_LP")
            .alias("VL_MEDIA_LP_ESTADO")
    )

)

In [0]:
df_municipio_silver = read(
    base_path=SILVER_PATH,
    table_name=TS_MUNICIPIO,
    format="delta"
).select(
    col("ANO_REFERENCIA"),
    col("CO_UF"),
    col("SG_UF"),
    col("REGIAO"),
    col("CO_MUNICIPIO"),
    col("NO_MUNICIPIO"),
    col("NO_MUNICIPIO_UF"),
    col("ID_TIPO_REDE"),
    col("DS_TIPO_REDE"),
    col("PC_ALUNO_ALFABETIZADO"),
    col("VL_MEDIA_LP"),
    col("FAIXA_MEDIA_LP"),
    col("FAIXA_ALFABETIZACAO")
)

In [0]:
META_ALFABETIZACAO = 80.0

df_indicador_municipio = (

    df_municipio_silver

    # =====================================================
    # Meta de Alfabetização
    # =====================================================

    .withColumn(
        "META_ALFABETIZACAO",
        lit(META_ALFABETIZACAO)
    )

    # =====================================================
    # Diferença para a Meta
    # =====================================================

    .withColumn(
        "DIF_META_ALFABETIZACAO",
        round(
            col("PC_ALUNO_ALFABETIZADO") - col("META_ALFABETIZACAO"),
            2
        )
    )

    # =====================================================
    # Atingiu a Meta
    # =====================================================

    .withColumn(
        "ATINGIU_META",
        when(
            col("PC_ALUNO_ALFABETIZADO") >= col("META_ALFABETIZACAO"),
            "Sim"
        ).otherwise("Não")
    )

    # =====================================================
    # Classificação do Município
    # =====================================================

    .withColumn(
        "CLASSIFICACAO",
        when(col("PC_ALUNO_ALFABETIZADO") >= 90, "Excelente")
        .when(col("PC_ALUNO_ALFABETIZADO") >= 80, "Muito Bom")
        .when(col("PC_ALUNO_ALFABETIZADO") >= 70, "Bom")
        .when(col("PC_ALUNO_ALFABETIZADO") >= 60, "Regular")
        .otherwise("Crítico")
    )

    # =====================================================
    # Ordem da Classificação
    # Facilita ordenação no Power BI
    # =====================================================

    .withColumn(
        "ORDEM_CLASSIFICACAO",
        when(col("CLASSIFICACAO") == "Excelente", 5)
        .when(col("CLASSIFICACAO") == "Muito Bom", 4)
        .when(col("CLASSIFICACAO") == "Bom", 3)
        .when(col("CLASSIFICACAO") == "Regular", 2)
        .otherwise(1)
    )

)

In [0]:
df_indicador_municipio_joined = (

    df_indicador_municipio.alias("m")

    .join(
        df_estado_indicador.alias("e"),
        on=[
            col("m.ANO_REFERENCIA") == col("e.ANO_REFERENCIA"),
            col("m.CO_UF") == col("e.CO_UF"),
            col("m.ID_TIPO_REDE") == col("e.ID_TIPO_REDE")
        ],
        how="left"
    )
    .drop(
        col("e.ANO_REFERENCIA"),
        col("e.CO_UF"),
        col("e.ID_TIPO_REDE")
    )

)

In [0]:
df_indicador_municipio_joined = (
    df_indicador_municipio_joined
    .withColumn(
        "DIF_ALFABETIZACAO_ESTADO",
        round(
            col("PC_ALUNO_ALFABETIZADO")
            -
            col("PC_ALUNO_ALFABETIZADO_ESTADO"),
            2
        )

    )
    .withColumn(
        "DIF_MEDIA_LP_ESTADO",
        round(
            col("VL_MEDIA_LP")
            -
            col("VL_MEDIA_LP_ESTADO"),
            2
        )

    )
    .withColumn(
        "ACIMA_MEDIA_ESTADO",
        when(
            col("DIF_ALFABETIZACAO_ESTADO") >= 0,
            "Sim"
        ).otherwise("Não")

    )

)

In [0]:
window_municipio = (
    Window
    .partitionBy(
        "CO_MUNICIPIO",
        "ID_TIPO_REDE"
    )
    .orderBy(
        "ANO_REFERENCIA"
    )

)

In [0]:
df_indicador_municipio_joined = (

    df_indicador_municipio_joined
    .withColumn(
        "VARIACAO_ALFABETIZACAO",
        round(
            col("PC_ALUNO_ALFABETIZADO")
            -
            lag(

                "PC_ALUNO_ALFABETIZADO"
            ).over(window_municipio),

            2
        )
    )

    .withColumn(
        "VARIACAO_MEDIA_LP",
        round(
            col("VL_MEDIA_LP")
            -
            lag(
                "VL_MEDIA_LP"
            ).over(window_municipio),
            2
        )
    )
)

In [0]:
df_indicador_municipio_joined = (
    df_indicador_municipio_joined
    .withColumn(
        "TENDENCIA",
        when(
            col("VARIACAO_ALFABETIZACAO") > 0,
            "Melhorou"
        )
        .when(
            col("VARIACAO_ALFABETIZACAO") < 0,
            "Piorou"
        )
        .when(
            col("VARIACAO_ALFABETIZACAO") == 0,
            "Estável"
        )
    )
)

In [0]:
write_delta(
    df=df_indicador_municipio_joined,
    base_path=GOLD_PATH,
    table_name=INDICADOR_MUNICIPIO,
    write_mode="overwrite"
)